# Exploratory Analysis: Dynamic Foraging Task

This notebook walks through the full analysis workflow for the two-option dynamic foraging experiment. It demonstrates:

1. **Data loading** — from Firebase or local files
2. **Validation** — quality checks and session exclusion
3. **Derived variables** — rolling windows, switching metrics, run lengths
4. **Trajectory plots** — individual and group-level behavioral time series
5. **Phase-transition analysis** — how behavior changes at regime shifts
6. **Model fitting** — baseline (bias, WSLS) and reinforcement-learning models
7. **Observed vs. simulated comparison** — model-checking via simulation

Each section is self-contained with explanatory commentary. See `outcome_definitions.md` for precise variable definitions.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Pipeline imports
from analysis_pipeline.utils import load_config, ensure_dirs
from analysis_pipeline.io import load_data
from analysis_pipeline.validate import validate_events
from analysis_pipeline.transform import compute_derived_variables
from analysis_pipeline.metrics import compute_session_metrics

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.facecolor": "white", "savefig.bbox": "tight"})

print("Setup complete.")

## 1. Load Configuration and Data

The pipeline is configured via `config.yaml`. Set `data_source` to `"firebase"` to pull live data from Firestore, or `"local"` to load from CSV/JSON files in the `data/` directory.

> **Tip:** If you haven't collected data yet, you can generate synthetic data with the cell in Section 1b below.

In [ ]:
config = load_config("config.yaml")
ensure_dirs(config.get("output_dir", "./outputs"))

# Display key config values
print("Data source:", config["data_source"])
print("Phase boundaries:")
for pb in config["phase_boundaries"]:
    print(f"  Phase {pb['id']}: {pb['start_ms']/1000:.0f}–{pb['end_ms']/1000:.0f}s  ({pb['label']})")

### 1b. Generate Synthetic Data (if no real data available yet)

The cell below simulates sessions that mimic the task's hidden-state dynamics. This lets you explore the full pipeline before deploying the experiment. Skip this cell if you already have real data.

In [ ]:
def simulate_session(session_id, participant_id, config, rng,
                     alpha=0.15, beta=4.0, forget=0.05):
    """
    Simulate one session using a Q-learning agent in the task environment.
    Returns a DataFrame of event-level data matching the real data schema.
    """
    phases = [
        {"id": 1, "start_ms": 0,      "end_ms": 90000,  "rA": 0.18, "rB": 0.18, "dA": 0.12, "dB": 0.12},
        {"id": 2, "start_ms": 90000,  "end_ms": 180000, "rA": 0.22, "rB": 0.10, "dA": 0.08, "dB": 0.15},
        {"id": 3, "start_ms": 180000, "end_ms": 270000, "rA": 0.10, "rB": 0.22, "dA": 0.15, "dB": 0.08},
        {"id": 4, "start_ms": 270000, "end_ms": 360000, "rA": 0.08, "rB": 0.08, "dA": 0.18, "dB": 0.18},
    ]
    # Bonus pulses in Phase 4
    pulse_onset_offsets = []
    pulse_seq = ["A", "B", "A", "B", "A", "B", "A", "B"]
    for i, target in enumerate(pulse_seq):
        onset = 270000 + i * 12000
        offset = onset + 3000
        pulse_onset_offsets.append((onset, offset, target))

    VA, VB = 0.7, 0.7
    QA, QB = 0.5, 0.5
    t_ms = 0
    cum_score = 0
    events = []

    while t_ms < 360000:
        # Determine current phase
        phase = phases[0]
        for p in phases:
            if p["start_ms"] <= t_ms < p["end_ms"]:
                phase = p
                break

        # Agent choice (softmax)
        dv = beta * (QA - QB)
        dv = np.clip(dv, -500, 500)
        p_a = 1.0 / (1.0 + np.exp(-dv))
        chose_a = rng.random() < p_a
        chosen = "A" if chose_a else "B"

        # ICI (sample from lognormal, ~0.3–2s, clipped)
        ici_ms = max(200, min(5000, int(rng.lognormal(np.log(500), 0.5))))
        t_ms += ici_ms

        if t_ms >= 360000:
            break

        # Recovery (continuous-time approximation)
        dt = ici_ms / 1000
        VA = min(1.0, VA + phase["rA"] * dt)
        VB = min(1.0, VB + phase["rB"] * dt)

        # Bonus pulse check
        bonus = 0.0
        active_pulse = None
        for onset, offset, target in pulse_onset_offsets:
            if onset <= t_ms < offset:
                active_pulse = target
                if target == chosen:
                    bonus = 0.20
                break

        # Reward
        v_chosen = VA if chose_a else VB
        reward_prob = min(1.0, v_chosen + bonus)
        rewarded = int(rng.random() < reward_prob)
        points = 10 if rewarded else 0
        cum_score += points

        # Depletion
        if chose_a:
            VA = max(0, VA - phase["dA"])
        else:
            VB = max(0, VB - phase["dB"])

        # Q-learning update
        r_signal = rewarded
        if chose_a:
            QA += alpha * (r_signal - QA)
            QB += forget * (0.5 - QB)
        else:
            QB += alpha * (r_signal - QB)
            QA += forget * (0.5 - QA)

        events.append({
            "participant_id": participant_id,
            "session_id": session_id,
            "timestamp_ms": t_ms,
            "click_index": len(events),
            "chosen_option": chosen,
            "reward_outcome": rewarded,
            "points_earned": points,
            "cumulative_score": cum_score,
            "time_since_prev_click_ms": ici_ms,
            "phase_id": phase["id"],
            "phase_label": config["phase_boundaries"][phase["id"] - 1]["label"],
            "latent_value_a_pre": VA + (phase["dA"] if chose_a else 0),  # pre-depletion
            "latent_value_b_pre": VB + (phase["dB"] if not chose_a else 0),
            "latent_value_a_post": VA,
            "latent_value_b_post": VB,
            "reward_probability_used": reward_prob,
            "active_bonus_pulse": 1 if active_pulse else 0,
            "bonus_target": active_pulse or "",
            "switch_flag": 0,  # will be recomputed by transform
        })

    return pd.DataFrame(events)


# ── Generate 8 synthetic sessions with varied parameters ──
rng = np.random.default_rng(42)
param_sets = [
    {"alpha": 0.10, "beta": 3.0, "forget": 0.02},
    {"alpha": 0.25, "beta": 5.0, "forget": 0.08},
    {"alpha": 0.05, "beta": 2.0, "forget": 0.01},
    {"alpha": 0.20, "beta": 6.0, "forget": 0.10},
    {"alpha": 0.15, "beta": 4.0, "forget": 0.05},
    {"alpha": 0.30, "beta": 3.5, "forget": 0.04},
    {"alpha": 0.08, "beta": 8.0, "forget": 0.03},
    {"alpha": 0.18, "beta": 4.5, "forget": 0.06},
]

sim_dfs = []
for i, params in enumerate(param_sets):
    sid = f"sim_session_{i+1:03d}"
    pid = f"sim_participant_{i+1:03d}"
    df = simulate_session(sid, pid, config, rng, **params)
    sim_dfs.append(df)

events_df = pd.concat(sim_dfs, ignore_index=True)
meta_df = pd.DataFrame()  # no metadata for synthetic data

print(f"Generated {events_df['session_id'].nunique()} synthetic sessions, "
      f"{len(events_df)} total clicks.")

## 2. Validation

The validation step checks for:
- Required columns present
- Monotonic timestamps and click indices
- Valid choice labels (A/B) and binary rewards
- Minimum click count and session duration

Sessions that fail hard criteria are excluded. Warnings are logged but don't trigger exclusion.

In [ ]:
events_df, validation_report = validate_events(events_df, config)

if len(validation_report) > 0:
    display(validation_report)
else:
    print("All sessions passed validation.")

print(f"\nRetained: {events_df['session_id'].nunique()} sessions, {len(events_df)} clicks")

## 3. Derived Variables

This step computes all event-level derived columns:
- **Choice indicators**: `choice_a`, `choice_b`
- **Switching**: `switch_flag_verified`, `run_length_current`, `run_length_previous`
- **Temporal**: `elapsed_time_s`, `ici_s`, `time_since_last_reward_s`, `time_since_last_switch_s`
- **Rolling windows**: choice proportion, reward rate, switch rate, ICI (click- and time-based)
- **Local reward estimates**: per-option reward rates within a rolling window
- **Win-stay / lose-shift**: trial-level reactive strategy indicators
- **Optimality**: `choice_optimal`, `regret_proxy`, `cumulative_regret` (requires latent values)

In [ ]:
events_df = compute_derived_variables(events_df, config)

print(f"Columns after transform: {len(events_df.columns)}")
print(f"\nDerived columns added:")
derived = [
    "choice_a", "switch_flag_verified", "run_length_current",
    "rolling_choice_prop_a_clicks", "rolling_reward_rate_clicks",
    "local_reward_rate_a", "local_reward_rate_b",
    "win_stay", "lose_shift", "choice_optimal", "cumulative_regret",
]
for col in derived:
    if col in events_df.columns:
        print(f"  {col}: {events_df[col].describe().round(3).to_dict()}")

## 4. Session-Level Metrics

Aggregates event-level data into one row per session, including per-phase breakdowns and adaptation lags at phase transitions.

In [ ]:
metrics_df = compute_session_metrics(events_df, config)

display(metrics_df[[
    "session_id", "total_clicks", "session_duration_s", "overall_reward_rate",
    "overall_switch_rate", "mean_run_length", "choice_entropy",
    "win_stay_rate", "lose_shift_rate",
]].round(3))

## 5. Trajectory Plots

### 5a. Individual Session — Full Multi-Panel View

We'll plot one example session to see the raw behavioral trajectory.

In [ ]:
# Pick one example session
example_sid = events_df["session_id"].unique()[0]
sdf = events_df[events_df["session_id"] == example_sid].copy()
phase_boundaries = config["phase_boundaries"]

def add_phase_lines(ax):
    for pb in phase_boundaries:
        if pb["start_ms"] > 0:
            ax.axvline(pb["start_ms"] / 1000, color="black", ls=":", alpha=0.4, lw=0.8)

fig = plt.figure(figsize=(16, 16))
gs = gridspec.GridSpec(5, 2, hspace=0.45, wspace=0.3)

# ── Panel 1: Choice raster ──
ax = fig.add_subplot(gs[0, :])
a_clicks = sdf[sdf["choice_a"] == 1]
b_clicks = sdf[sdf["choice_b"] == 1]
ax.scatter(a_clicks["elapsed_time_s"], [1]*len(a_clicks), s=4, color="dodgerblue", marker="|", label="A")
ax.scatter(b_clicks["elapsed_time_s"], [0]*len(b_clicks), s=4, color="orangered", marker="|", label="B")
add_phase_lines(ax)
ax.set_yticks([0, 1]); ax.set_yticklabels(["B", "A"])
ax.set_xlabel("Time (s)"); ax.set_title(f"Session {example_sid}: Choice Raster"); ax.legend(fontsize=9)

# ── Panel 2: Rolling choice proportion ──
ax = fig.add_subplot(gs[1, :])
ax.plot(sdf["elapsed_time_s"], sdf["rolling_choice_prop_a_clicks"], color="steelblue", lw=1.2, label="Click-window")
ax.plot(sdf["elapsed_time_s"], sdf["rolling_choice_prop_a_time"], color="coral", lw=1, alpha=0.7, label="Time-window")
ax.axhline(0.5, color="gray", ls="--", alpha=0.5)
add_phase_lines(ax)
ax.set_ylim(0, 1); ax.set_ylabel("P(Choose A)"); ax.set_xlabel("Time (s)")
ax.set_title("Rolling Choice Proportion"); ax.legend(fontsize=9)

# ── Panel 3: Cumulative score ──
ax = fig.add_subplot(gs[2, 0])
ax.plot(sdf["elapsed_time_s"], sdf["cumulative_score"], color="steelblue")
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Cumulative Score"); ax.set_title("Score Trajectory")

# ── Panel 4: Rolling reward rate ──
ax = fig.add_subplot(gs[2, 1])
ax.plot(sdf["elapsed_time_s"], sdf["rolling_reward_rate_clicks"], color="darkorange", alpha=0.8)
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Reward Rate"); ax.set_title("Rolling Reward Rate")

# ── Panel 5: Latent values ──
ax = fig.add_subplot(gs[3, 0])
ax.plot(sdf["elapsed_time_s"], sdf["latent_value_a_pre"], color="dodgerblue", alpha=0.7, label="V_A")
ax.plot(sdf["elapsed_time_s"], sdf["latent_value_b_pre"], color="orangered", alpha=0.7, label="V_B")
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Latent Value"); ax.set_title("Hidden State Dynamics"); ax.legend(fontsize=8)

# ── Panel 6: Local reward rate per option ──
ax = fig.add_subplot(gs[3, 1])
ax.plot(sdf["elapsed_time_s"], sdf["local_reward_rate_a"], color="dodgerblue", alpha=0.7, label="Rate A")
ax.plot(sdf["elapsed_time_s"], sdf["local_reward_rate_b"], color="orangered", alpha=0.7, label="Rate B")
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Local Reward Rate"); ax.set_title("Experienced Reward Rate"); ax.legend(fontsize=8)

# ── Panel 7: Run lengths ──
ax = fig.add_subplot(gs[4, 0])
ax.scatter(sdf["elapsed_time_s"], sdf["run_length_current"], s=5, alpha=0.4, color="teal")
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Run Length"); ax.set_title("Consecutive Same-Option Runs")

# ── Panel 8: Cumulative regret ──
ax = fig.add_subplot(gs[4, 1])
ax.plot(sdf["elapsed_time_s"], sdf["cumulative_regret"], color="crimson")
add_phase_lines(ax)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Cumulative Regret"); ax.set_title("Cumulative Regret")

plt.suptitle(f"Individual Session Dashboard: {example_sid}", fontsize=14, y=1.01)
plt.show()

### 5b. Group Trajectory Overlay

Overlay all sessions' rolling choice proportion to visualize population-level trends and individual variability.

In [ ]:
edf = events_df.copy()
edf["time_bin"] = (edf["elapsed_time_s"] // 1).astype(int)

fig, ax = plt.subplots(figsize=(14, 5))

# Individual traces
for sid, sdf in edf.groupby("session_id"):
    binned = sdf.groupby("time_bin")["rolling_choice_prop_a_clicks"].mean()
    ax.plot(binned.index, binned.values, alpha=0.25, lw=0.7, color="steelblue")

# Group mean +/- SEM
group = edf.groupby("time_bin")["rolling_choice_prop_a_clicks"].agg(["mean", "sem"]).dropna()
ax.plot(group.index, group["mean"], color="darkblue", lw=2.5, label="Group mean")
ax.fill_between(group.index, group["mean"] - group["sem"], group["mean"] + group["sem"],
                alpha=0.2, color="steelblue")

ax.axhline(0.5, color="gray", ls="--", alpha=0.5)
add_phase_lines(ax)

# Phase annotations
for pb in phase_boundaries:
    mid = (pb["start_ms"] + pb["end_ms"]) / 2000
    ax.text(mid, 0.02, pb["label"], ha="center", fontsize=8, color="gray", style="italic")

ax.set_xlim(0, 360); ax.set_ylim(0, 1)
ax.set_xlabel("Time (s)"); ax.set_ylabel("P(Choose A)")
ax.set_title("Choice Proportion Trajectories — All Sessions"); ax.legend()
plt.show()

## 6. Phase-Transition Analysis

How does behavior change when the environment shifts? We align data around each phase boundary and compare pre- vs post-transition choice proportions.

The dotted red line marks the transition. In Phase 2, the A option becomes advantaged, so P(A) should rise. In Phase 3, it reverses.

In [ ]:
pre_s = config.get("transition_pre_window_s", 15)
post_s = config.get("transition_post_window_s", 30)

transitions = [
    {"label": "Phase 1 -> 2 (A advantage)", "time_s": 90},
    {"label": "Phase 2 -> 3 (B advantage)", "time_s": 180},
    {"label": "Phase 3 -> 4 (Scarcity)",    "time_s": 270},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, tr in zip(axes, transitions):
    t = tr["time_s"]
    window = events_df[
        (events_df["elapsed_time_s"] >= t - pre_s) &
        (events_df["elapsed_time_s"] <= t + post_s)
    ].copy()
    window["rel_time"] = window["elapsed_time_s"] - t
    window["time_bin"] = (window["rel_time"] // 1).astype(int)

    # Individual traces
    for sid, sdf in window.groupby("session_id"):
        binned = sdf.groupby("time_bin")["choice_a"].mean()
        ax.plot(binned.index, binned.values, alpha=0.2, lw=0.5, color="steelblue")

    # Group mean
    group = window.groupby("time_bin")["choice_a"].mean()
    ax.plot(group.index, group.values, color="darkblue", lw=2.5)

    ax.axvline(0, color="red", ls="--", alpha=0.7, label="Transition")
    ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
    ax.set_xlabel("Time relative to transition (s)")
    ax.set_ylabel("P(Choose A)" if ax == axes[0] else "")
    ax.set_title(tr["label"])
    ax.set_ylim(0, 1)

axes[0].legend(fontsize=8)
plt.suptitle("Behavior Around Phase Transitions", fontsize=13)
plt.tight_layout()
plt.show()

# ── Pre vs Post paired t-tests ──
print("\nPre-Post Paired t-tests (choice proportion):")
print("-" * 70)
for tr in transitions:
    t = tr["time_s"]
    pre_vals, post_vals = [], []
    for sid, sdf in events_df.groupby("session_id"):
        pre = sdf[(sdf["elapsed_time_s"] >= t - pre_s) & (sdf["elapsed_time_s"] < t)]
        post = sdf[(sdf["elapsed_time_s"] >= t) & (sdf["elapsed_time_s"] < t + post_s)]
        if len(pre) > 0 and len(post) > 0:
            pre_vals.append(pre["choice_a"].mean())
            post_vals.append(post["choice_a"].mean())
    if len(pre_vals) >= 3:
        t_stat, p_val = stats.ttest_rel(pre_vals, post_vals)
        diff = np.mean(np.array(post_vals) - np.array(pre_vals))
        print(f"  {tr['label']}:  diff = {diff:+.3f},  t = {t_stat:.2f},  p = {p_val:.4f}")

## 7. Model Fitting

We fit two families of models to each session's choice sequence:

1. **Baseline models** (bias, WSLS) — simple heuristics
2. **RL models** (Q-learning, dual-alpha, forgetting) — value-based learning

Models are compared via BIC (lower = better). We then inspect recovered parameters.

In [ ]:
from analysis_pipeline.models.baseline_models import fit_all_baselines
from analysis_pipeline.models.rl_models import fit_rl_models

# Fit baseline models
print("Fitting baseline models...")
baseline_results = fit_all_baselines(events_df)
print(f"  {len(baseline_results)} fits done.")

# Fit RL models
print("Fitting RL models...")
rl_results = fit_rl_models(events_df, config)
print(f"  {len(rl_results)} fits done.")

# Combine
all_models = pd.concat([baseline_results, rl_results], ignore_index=True)

# ── BIC comparison table (mean across sessions) ──
model_summary = all_models.groupby("model").agg(
    mean_bic=("bic", "mean"),
    mean_aic=("aic", "mean"),
    mean_nll=("nll", "mean"),
    n_sessions=("session_id", "nunique"),
).round(1).sort_values("mean_bic")

print("\nModel Comparison (mean across sessions, sorted by BIC):")
display(model_summary)

In [ ]:
# ── BIC by model, per session (heatmap-style) ──
bic_pivot = all_models.pivot(index="session_id", columns="model", values="bic")

fig, ax = plt.subplots(figsize=(10, 5))
# Normalize each row to highlight the best model
bic_norm = bic_pivot.sub(bic_pivot.min(axis=1), axis=0)  # delta-BIC from best
sns.heatmap(bic_norm, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "Delta BIC from best"})
ax.set_title("Model Comparison: Delta-BIC per Session (0 = best)")
ax.set_ylabel("Session")
plt.tight_layout()
plt.show()

# ── Recovered RL parameters ──
rl_params = rl_results[rl_results["model"] == "q_forgetting"].copy()
if len(rl_params) > 0:
    print("\nRecovered Q-Learning (forgetting) parameters:")
    for _, row in rl_params.iterrows():
        print(f"  {row['session_id']}: alpha={row.get('alpha', 'N/A'):.3f}, "
              f"beta={row.get('beta', 'N/A'):.2f}, "
              f"forget={row.get('forget', 'N/A'):.3f}")

## 8. Observed vs. Simulated Comparison

The strongest model check: use the best-fit RL parameters to **simulate** new choice sequences through the same environment, then compare the simulated trajectories to the observed data.

If the model captures the generative process, simulated and observed trajectories should share the same distributional properties (choice proportions, switch rates, run-length distributions, adaptation dynamics).

In [ ]:
# ── Simulate from best-fit forgetting Q-learning parameters ──

def simulate_from_params(observed_sdf, alpha, beta, forget, rng, n_sims=20):
    """
    Re-simulate choice sequences using the observed session's timestamps
    and environment (latent values, phases), but with the model generating choices.
    Returns a list of rolling-choice-prop arrays (one per simulation).
    """
    timestamps = observed_sdf["timestamp_ms"].values
    rewards_obs = observed_sdf["reward_outcome"].values
    va_pre = observed_sdf["latent_value_a_pre"].values
    vb_pre = observed_sdf["latent_value_b_pre"].values
    n = len(timestamps)
    window = 20

    sim_traces = []
    for _ in range(n_sims):
        QA, QB = 0.5, 0.5
        choices = np.zeros(n, dtype=int)

        for t in range(n):
            # Softmax choice
            dv = np.clip(beta * (QA - QB), -500, 500)
            p_a = 1.0 / (1.0 + np.exp(-dv))
            chose_a = int(rng.random() < p_a)
            choices[t] = chose_a

            # Reward from actual latent values (same environment)
            v_chosen = va_pre[t] if chose_a else vb_pre[t]
            rewarded = int(rng.random() < min(1.0, v_chosen))

            # Update
            if chose_a:
                QA += alpha * (rewarded - QA)
                QB += forget * (0.5 - QB)
            else:
                QB += alpha * (rewarded - QB)
                QA += forget * (0.5 - QA)

        # Compute rolling prop
        rolling = pd.Series(choices).rolling(window, min_periods=1).mean().values
        sim_traces.append(rolling)

    return sim_traces


# ── Run simulation for the example session ──
example_sid = events_df["session_id"].unique()[0]
obs_sdf = events_df[events_df["session_id"] == example_sid].sort_values("timestamp_ms")

# Get that session's best-fit parameters
sid_params = rl_results[
    (rl_results["session_id"] == example_sid) &
    (rl_results["model"] == "q_forgetting")
]

if len(sid_params) > 0:
    params = sid_params.iloc[0]
    fit_alpha = params["alpha"]
    fit_beta = params["beta"]
    fit_forget = params["forget"]

    rng_sim = np.random.default_rng(123)
    sim_traces = simulate_from_params(obs_sdf, fit_alpha, fit_beta, fit_forget,
                                       rng_sim, n_sims=50)

    # ── Plot: observed vs simulated trajectories ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    elapsed = obs_sdf["elapsed_time_s"].values

    # Left: trajectory comparison
    ax = axes[0]
    for trace in sim_traces:
        ax.plot(elapsed, trace, alpha=0.08, color="coral", lw=0.5)
    ax.plot(elapsed, obs_sdf["rolling_choice_prop_a_clicks"].values,
            color="darkblue", lw=2, label="Observed")
    # Simulated mean
    sim_mean = np.mean(sim_traces, axis=0)
    ax.plot(elapsed, sim_mean, color="red", lw=2, ls="--", label="Simulated mean")

    add_phase_lines(ax)
    ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("P(Choose A)")
    ax.set_title(f"Observed vs Simulated Trajectories\n(alpha={fit_alpha:.3f}, "
                 f"beta={fit_beta:.2f}, forget={fit_forget:.3f})")
    ax.legend()

    # Right: run-length distributions
    ax = axes[1]

    # Observed run lengths
    obs_choices = obs_sdf["choice_a"].values
    obs_runs = []
    run = 1
    for i in range(1, len(obs_choices)):
        if obs_choices[i] == obs_choices[i-1]:
            run += 1
        else:
            obs_runs.append(run)
            run = 1
    obs_runs.append(run)

    # Simulated run lengths (pool all simulations)
    sim_runs = []
    for trace_choices_raw in sim_traces:
        # We need the actual binary choices, not the rolling mean
        pass

    # Re-simulate to get raw choices for run-length comparison
    sim_all_runs = []
    for _ in range(50):
        QA, QB = 0.5, 0.5
        choices_sim = []
        va_pre = obs_sdf["latent_value_a_pre"].values
        vb_pre = obs_sdf["latent_value_b_pre"].values
        for t in range(len(va_pre)):
            dv = np.clip(fit_beta * (QA - QB), -500, 500)
            p_a = 1.0 / (1.0 + np.exp(-dv))
            chose_a = int(rng_sim.random() < p_a)
            choices_sim.append(chose_a)
            v_chosen = va_pre[t] if chose_a else vb_pre[t]
            rewarded = int(rng_sim.random() < min(1.0, v_chosen))
            if chose_a:
                QA += fit_alpha * (rewarded - QA)
                QB += fit_forget * (0.5 - QB)
            else:
                QB += fit_alpha * (rewarded - QB)
                QA += fit_forget * (0.5 - QA)

        run = 1
        for i in range(1, len(choices_sim)):
            if choices_sim[i] == choices_sim[i-1]:
                run += 1
            else:
                sim_all_runs.append(run)
                run = 1
        sim_all_runs.append(run)

    max_run = max(max(obs_runs), np.percentile(sim_all_runs, 95))
    bins = np.arange(1, min(int(max_run) + 2, 30))
    ax.hist(obs_runs, bins=bins, density=True, alpha=0.6, color="steelblue",
            edgecolor="white", label="Observed")
    ax.hist(sim_all_runs, bins=bins, density=True, alpha=0.4, color="coral",
            edgecolor="white", label="Simulated")
    ax.set_xlabel("Run Length"); ax.set_ylabel("Density")
    ax.set_title("Run-Length Distribution: Observed vs Simulated")
    ax.legend()

    plt.tight_layout()
    plt.show()

    # ── Summary statistics comparison ──
    print("\nObserved vs Simulated Summary:")
    print(f"  Observed  — mean run: {np.mean(obs_runs):.2f}, "
          f"switch rate: {obs_sdf['switch_flag_verified'].mean():.3f}, "
          f"P(A): {obs_sdf['choice_a'].mean():.3f}")
    print(f"  Simulated — mean run: {np.mean(sim_all_runs):.2f}, "
          f"P(A): {np.mean([np.mean(t) for t in sim_traces]):.3f}")
else:
    print("No forgetting Q-learning fit found for this session.")

## 9. Summary and Next Steps

This notebook demonstrated the core analysis workflow:

| Step | What it does |
|------|-------------|
| **Load & validate** | Ingest data, check quality, exclude bad sessions |
| **Transform** | Compute 30+ derived behavioral variables |
| **Metrics** | Aggregate to session-level summaries |
| **Trajectory plots** | Visualize individual and group choice dynamics |
| **Phase analysis** | Quantify adaptation at regime shifts |
| **Model fitting** | Compare heuristic vs. RL accounts of behavior |
| **Simulation** | Model-checking via observed-vs-simulated comparison |

### To run the full pipeline on real data:

```bash
source venv/bin/activate
python run_analysis.py --config config.yaml
```

### Additional analyses available in the pipeline (not shown here):
- **Perturbation analysis** — pulse-triggered averaging of Phase 4 bonus responses
- **Dynamical analysis** — state-space trajectories, hysteresis, autocorrelation, recurrence
- **Matching law** — generalized matching sensitivity and bias
- **HTML report** — auto-generated summary with all figures and tables

See `outcome_definitions.md` for precise definitions of every variable computed above.